In [ ]:
import numpy as np
import json
import pandas as pd
from datetime import datetime
import random
import pycountry
import arviz as az
from IPython.display import display, Markdown

from emu_renewal.constants import DATA_PATH, RERUNS, FULL_RUN, TIMEOUTS, DATE_FORMAT, BASE_PATH
from emu_renewal.inputs import get_cont_of_country
from emu_renewal.run import run_identifiability
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_input_recovery
from emu_renewal.utils import get_analysis_paths

In [ ]:
n_iters = 10000
n_analyses = 5
np.random.seed(0)

all_countries = json.load(open(DATA_PATH / "config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries)
eligible_countries = [c for c in analysis_paths if "fb_visited_mob" in analysis_paths[c] and get_cont_of_country(c) != "OC"]
sample_countries = random.sample(eligible_countries, n_analyses)
analysis_time = datetime.now().strftime(DATE_FORMAT)
analysis_types = ["fb_visited_mob", "fb_singletile_mob"]
analyses = [analysis_types[i] for i in np.random.choice(len(analysis_types), n_analyses)]

In [ ]:
scalar_params = {}
multi_params = {}
for a in range(n_analyses):
    iso3 = sample_countries[a]
    mob_source = analyses[a]
    display(Markdown(f"\n country: {pycountry.countries.lookup(iso3).name}"))
    display(Markdown(f"mobility approach: {mob_source}"))

    # Select random set of scalar parameters from calibration posterior
    path = analysis_paths[iso3][mob_source]
    idata = az.from_netcdf(path / "idata_filtered.nc")
    post = idata.posterior
    chain = np.random.randint(post.sizes["chain"])
    draw = np.random.randint(post.sizes["draw"])
    draw_params = post.isel(chain=chain, draw=draw)
    scalar_params[iso3] = {p: float(v) for p, v in draw_params.items() if v.ndim == 0}
    multi_params[iso3] = {p: v.values for p, v in draw_params.items() if v.ndim > 0}
    
    # Run analysis
    run_identifiability(iso3, mob_source, analysis_time, scalar_params[iso3], multi_params[iso3], n_iters)

In [ ]:
for a in range(n_analyses):
    iso3 = sample_countries[a]
    mob_source = analyses[a]
    display(Markdown(f"\n country: {pycountry.countries.lookup(iso3).name}"))
    display(Markdown(f"mobility approach: {mob_source}"))

    
    path = BASE_PATH / "identify_outputs" / analysis_time / iso3 / mob_source
    idata = az.from_netcdf( path/ "idata_filtered.nc")
    targets = load_targets(path)
    spaghetti = pd.read_hdf(path / "spaghetti.h5")
    updates = pd.read_hdf(path / "updates.h5")

    display(plot_input_recovery(scalar_params[iso3], multi_params[iso3], idata, targets, spaghetti, updates))